In [1]:
!pip install pandas scikit-learn

In [3]:
import pandas as pd
import torch
import numpy as np
import pickle
import os
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. CONFIGURATION ---
NODES_CSV_PATH = "merged_nodes.csv"
VECTORS_SAVE_PATH = "kg_vectors.npy"
NAMES_SAVE_PATH = "kg_names.pkl"
MODEL_NAME = "FremyCompany/BioLORD-2023"
THRESHOLD = 0.75

class SmartMedicalMatcher:
    def __init__(self, csv_path, vectors_path, names_path):
        print(f"--- Initializing Smart Matcher ---")
        
        # 1. Load Model (needed for matching even if space is loaded)
        self.embeddings_model = HuggingFaceEmbeddings(
            model_name=MODEL_NAME,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )

        # 2. Check if pre-saved semantic space exists
        if os.path.exists(vectors_path) and os.path.exists(names_path):
            print("📂 Loading pre-saved semantic space from disk...")
            self.kg_vectors = np.load(vectors_path)
            with open(names_path, 'rb') as f:
                self.kg_names = pickle.load(f)
            print(f"✅ Loaded {len(self.kg_names)} nodes successfully.")
        else:
            print("🔍 No saved space found. Processing CSV...")
            self._build_and_save_space(csv_path, vectors_path, names_path)

    def _build_and_save_space(self, csv_path, vectors_path, names_path):
        # Read KG Nodes
        df = pd.read_csv(csv_path, encoding='latin-1')
        target_col = 'source_name' if 'source_name' in df.columns else df.columns[0]
        self.kg_names = df[target_col].astype(str).tolist()
        
        # Encode
        print(f"🧠 Encoding {len(self.kg_names)} nodes (this happens only once)...")
        self.kg_vectors = np.array(self.embeddings_model.embed_documents(self.kg_names))
        
        # Save to disk
        np.save(vectors_path, self.kg_vectors)
        with open(names_path, 'wb') as f:
            pickle.dump(self.kg_names, f)
        print(f"💾 Semantic space saved to {vectors_path} and {names_path}")

    def match(self, emr_term: str):
        # Logic remains the same
        abbrev_map = {"hb": "hemoglobin", "uti": "urinary tract infection", "bp": "blood pressure", "mi": "myocardial infarction"}
        term = emr_term.lower().strip()
        for abbrev, full in abbrev_map.items():
            if abbrev == term or abbrev in term.split():
                term = term.replace(abbrev, full)

        term_vector = np.array(self.embeddings_model.embed_query(term)).reshape(1, -1)
        similarities = cosine_similarity(term_vector, self.kg_vectors)[0]
        best_idx = np.argmax(similarities)
        score = similarities[best_idx]

        if score >= THRESHOLD:
            return {"term": emr_term, "match": self.kg_names[best_idx], "conf": round(float(score), 3)}
        return {"term": emr_term, "match": "UNRESOLVED", "conf": round(float(score), 3)}

# --- EXECUTION ---
if __name__ == "__main__":
    # The first time you run this, it will encode. 
    # The second time, it will load in milliseconds.
    matcher = SmartMedicalMatcher(NODES_CSV_PATH, VECTORS_SAVE_PATH, NAMES_SAVE_PATH)

    test_cases = ["heart attack", "hb low", "kidney fail", "glucose high", "uti info"]
    
    print("\n" + "="*60)
    print(f"{'EMR TERM':<20} | {'BIO-LORD MATCH':<25} | {'CONF'}")
    print("-" * 60)
    for t in test_cases:
        res = matcher.match(t)
        print(f"{res['term']:<20} | {res['match']:<25} | {res['conf']}")
    print("="*60)

--- Initializing Smart Matcher ---
🔍 No saved space found. Processing CSV...
🧠 Encoding 9843 nodes (this happens only once)...
💾 Semantic space saved to kg_vectors.npy and kg_names.pkl

EMR TERM             | BIO-LORD MATCH            | CONF
------------------------------------------------------------
heart attack         | Myocardial Infarction     | 0.786
hb low               | Hemoglobin Low            | 1.0
kidney fail          | Renal Failure             | 0.976
glucose high         | Serum Glucose High        | 0.986
uti info             | Urinary Infection         | 0.872
